# Wiki

In [3]:
!pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers

In [4]:
import os
import json
from typing import List
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Indexing

In [5]:
from dotenv import load_dotenv
load_dotenv()

False

### [1] Load

In [9]:
from glob import glob
import json

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

docs = []

# 2) glossary json 불러오기
with open("./data/f1_glossary_all.json", "r", encoding="utf-8") as f:
    glossary_data = json.load(f)

for i, item in enumerate(glossary_data):
    term = item.get("term", "").strip()
    desc = item.get("description", "").strip()

    if term and desc:
        docs.append(
            Document(
                page_content=f"Term: {term}\nDescription: {desc}",
                metadata={
                    "source": "f1_glossary_all.json",
                    "doc_type": "glossary",
                    "row": i,
                    "term": term,
                },
            )
        )

print("glossary 추가 후 문서 수:", len(docs))

# 3) history wiki json 불러오기
with open("./data/f1_history_wiki.json", "r", encoding="utf-8") as f:
    wiki_data = json.load(f)

if isinstance(wiki_data, str):
    wiki_text = wiki_data
else:
    wiki_text = json.dumps(wiki_data, ensure_ascii=False)

docs.append(
    Document(
        page_content=wiki_text,
        metadata={
            "source": "f1_history_ko_wiki.json",
            "doc_type": "wiki",
        },
    )
)

print("wiki 추가 후 문서 수:", len(docs))

print("최종 원본 문서 수:", len(docs))

glossary 추가 후 문서 수: 244
wiki 추가 후 문서 수: 245
최종 원본 문서 수: 245


### [2] Splitter

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

glossary_docs = []
long_docs = []

for doc in docs:
    if doc.metadata.get("doc_type") == "glossary":
        glossary_docs.append(doc)
    else:
        long_docs.append(doc)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""]
)

split_long_docs = splitter.split_documents(long_docs)

docs = glossary_docs + split_long_docs

print("glossary 문서 수:", len(glossary_docs))
print("긴 문서 chunk 수:", len(split_long_docs))
print("최종 chunk 수:", len(docs))

glossary 문서 수: 244
긴 문서 chunk 수: 535
최종 chunk 수: 779


### [3] Embed

In [11]:
!pip install -U langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.8 MB/s eta 0:00:00


In [13]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [14]:
from langchain_openai.embeddings import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

### [4] Store

In [15]:
import os
import shutil
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1) 임베딩 모델
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# 2) 완전히 새 경로 사용
persist_dir = "/content/chroma_f1_db_new"

# 기존 폴더가 있으면 강제 삭제
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir, ignore_errors=True)

os.makedirs(persist_dir, exist_ok=True)

# 3) 새 컬렉션명으로 생성
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=persist_dir,
    collection_name="f1_rules_v2"
)

# 2. 생성

### [1] Prompt 생성 + Retrieval Chain

In [ ]:
# from langchain.chat_models import init_chat_model
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_classic.chains import create_retrieval_chain
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# llm = init_chat_model(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     model_provider="huggingface",
#     temperature=0.1,
#     max_tokens=512,
# )

# prompt = ChatPromptTemplate.from_messages([
#     (
#         "system",
#         """당신은 F1 문서 기반 질의응답 시스템입니다.
# 제공된 context만 참고해서 답변하세요.
# context에서 확인할 수 없는 내용은 추론하지 말고
# '문서에서 확인되지 않습니다.'라고 답하세요.
# 답변은 한국어로 간결하고 정확하게 작성하세요."""
#     ),
#     (
#         "human",
#         """질문: {input}

# context:
# {context}

# 답변:"""
#     )
# ])

# retriever = vector_store.as_retriever(search_kwargs={"k": 4})
# qa_chain = create_stuff_documents_chain(llm, prompt)
# rag_chain = create_retrieval_chain(retriever, qa_chain)

### test_case 생성

In [16]:
import random
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_id, dtype = torch.bfloat16, device_map = 'auto')
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False, #핵심
)

hf = HuggingFacePipeline(pipeline=pipe)
llm = ChatHuggingFace(llm=hf)

def generate_test_cases_per_source(vector_store, n_per_source=20):
    """PDF 출처별로 test case를 n개씩 생성"""

    # 1. 전체 청크 꺼내기 (메타데이터 포함)
    all_data = vector_store.get(include=["documents", "metadatas"])
    documents = all_data['documents']
    metadatas = all_data['metadatas']

    # 2. 출처(source)별로 청크 그룹화
    source_groups = {}
    for doc, meta in zip(documents, metadatas):
        source = meta.get('source', 'unknown')
        if source not in source_groups:
            source_groups[source] = []
        source_groups[source].append(doc)

    print(f"발견된 PDF 파일: {list(source_groups.keys())}")

    all_test_cases = {}

    for source, chunks in source_groups.items():
        print(f"\n=== {source} 처리 중 ({len(chunks)}개 청크) ===")

        # 3. 각 PDF에서 n개 랜덤 샘플링
        sampled = random.sample(chunks, min(n_per_source, len(chunks)))
        test_cases = []

        for i, chunk in enumerate(sampled):
            print(f"  [{i+1}/{n_per_source}] 생성 중...")

            prompt = f"""다음 텍스트를 바탕으로 Q&A를 1개 만들어줘.
텍스트에 답이 명확하게 있는 질문이어야 해.
한국어로 작성해줘.

텍스트:
{chunk}

아래 형식으로만 답해줘 (다른 말 하지 마):
질문: ...
정답: ...
"""
            response = llm.invoke(prompt).content
            lines = response.strip().split('\n')

            try:
                question = lines[0].replace("질문: ", "").strip()
                answer = lines[1].replace("정답: ", "").strip()
                test_cases.append({
                    "question": question,
                    "expected_answer": answer,
                    "source": source
                })
            except:
                print(f"  파싱 실패, 스킵")
                continue

        all_test_cases[source] = test_cases
        print(f"  → {len(test_cases)}개 생성 완료")

    return all_test_cases


# 실행 (문서당 20개)
all_test_cases = generate_test_cases_per_source(vector_store, n_per_source=20)

# 결과 확인
for source, cases in all_test_cases.items():
    print(f"\n[{source}] - {len(cases)}개")
    for i, tc in enumerate(cases[:3]):  # 앞에 3개만 미리보기
        print(f"  Q{i+1}: {tc['question']}")
        print(f"  A{i+1}: {tc['expected_answer']}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

  [18/20] 생성 중...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [19/20] 생성 중...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [20/20] 생성 중...
  → 20개 생성 완료

[f1_glossary_all.json] - 20개
  Q1: 체크터드 플래그는 어떤 의미를 가지며, 언제 사용됩니까?
  A1: 체크터드 플래그는 세션의 종료를 알리는 검은색과 백색의 체크무늬 패턴의 플래그입니다. 이 플래그는 포트 월 위에 끝선 바로 앞에서 휘두르되, 여전히 경주 중인 모든 차량이 플래그를 지나가기까지 계속됩니다.
  Q2: ERS의 주요 구성 요소는 무엇인가요?
  A2: ERS의 주요 구성 요소는 Motor Generator Unit - Kinetic (MGU-K), 에너지 저장 장치, 그리고 제어 전자기기입니다.
  Q3: 불법 주행을 반복할 경우 어떤 표시가 주행자에게 경고를 주나요?
  A3: 대각선으로 나누어진 검은색과 백색의 깃발이 주행자에게 불법 주행을 반복하면 처벌을 받을 수 있다는 경고를 줍니다.

[f1_history_ko_wiki.json] - 20개
  Q1: 1957년부터 1961년까지의 포뮬러 원은 어떤 변화를 겪었나요?
  A1: 1957년부터 1961년까지의 포뮬러 원은 기술적인 산업 제조업체들의 무작위 참가에서 전략적으로 경쟁적인 비즈니스로 변모했습니다.
  Q2: 1979년에 출전한 팀 중 예상치 못하게 어떤 팀이 포함되어 있었습니까?
  A2: Ferrari
  Q3: 뱅크로브 그랑프리에서 맥라렌은 어떤 변화를 통해 레이스에서 승리를 거두었나요?
  A3: 맥라렌은 독일 그랑프리에서 업그레이드를 통해 승리를 거두었습니다.


In [17]:
for source, cases in all_test_cases.items():
    print(f"\n[{source}] - {len(cases)}개")
    for i, tc in enumerate(cases):  # 앞에 3개만 미리보기
        print(f"  Q{i+1}: {tc['question']}")
        print(f"  A{i+1}: {tc['expected_answer']}")


[f1_glossary_all.json] - 20개
  Q1: 체크터드 플래그는 어떤 의미를 가지며, 언제 사용됩니까?
  A1: 체크터드 플래그는 세션의 종료를 알리는 검은색과 백색의 체크무늬 패턴의 플래그입니다. 이 플래그는 포트 월 위에 끝선 바로 앞에서 휘두르되, 여전히 경주 중인 모든 차량이 플래그를 지나가기까지 계속됩니다.
  Q2: ERS의 주요 구성 요소는 무엇인가요?
  A2: ERS의 주요 구성 요소는 Motor Generator Unit - Kinetic (MGU-K), 에너지 저장 장치, 그리고 제어 전자기기입니다.
  Q3: 불법 주행을 반복할 경우 어떤 표시가 주행자에게 경고를 주나요?
  A3: 대각선으로 나누어진 검은색과 백색의 깃발이 주행자에게 불법 주행을 반복하면 처벌을 받을 수 있다는 경고를 줍니다.
  Q4: 팀들이 사용할 수 있는 윈드 터널 시간은 어떻게 결정됩니까?
  A4: 팀들이 사용할 수 있는 윈드 터널 시간은 전년도 챔피언십에서의 성적에 따라 제한됩니다.
  Q5: F1 팀에서 DEVELOPMENT DRIVER의 주요 업무는 무엇인가요?
  A5: F1 팀에서 DEVELOPMENT DRIVER의 주요 업무는 차량 개발을 돕는 것입니다. 이는 일반적으로 시뮬레이터에서 시간을 보내는 것을 포함하며, 트랙에서의 테스트 시간은 제한적입니다.
  Q6: 드라이버의 헬멧 빌더에 부착된 테어오프 스trip는 무엇인가요?
  A6: 투명한 필름으로 만든 얇은 조각입니다.
  Q7: 헬멧에 장착되는 VISOR의 주요 기능은 무엇인가요?
  A7: 헬멧에 장착되는 VISOR는 운전자의 눈을 보호하는 투명 화면이며, 밝은 빛을更好地翻译成英文：
  Q8: F1 차량에서 엔진이 쇠퇴하는 원인은 무엇인가요?
  A8: 클러치 패들 문제 또는 렉스가 너무 낮아지는 기술적 문제로 인해 엔진이 쇠퇴할 수 있습니다.
  Q9: 한 시즌 동안 드라이버에게 주어지는 최대 재판단 횟수는 몇 번인가요?
  A9: 최대 네 번입니다.
  Q10: 드라이